# Load and Clean Downloaded FX Price Data

Scratch notebook for pulling raw daily candles from `fx_candles.db` and eyeballing
recent data before building indicators on top of it.

- Fetches every stored daily candle for `INSTRUMENT` over the 10-year window
  `[START_YEAR, END_YEAR)`, i.e. up to but not including `END_YEAR`.
- Prints full detail for the most recent `LAST_N_DAYS` trading days.
- Re-run all cells to refresh against the latest data in `fx_candles.db`.

In [ ]:
from __future__ import annotations

import sqlite3
from typing import Final

import pandas as pd

In [ ]:
DB_PATH: Final[str] = "../../fx_candles.db"
TABLE: Final[str] = "candles_D"
INSTRUMENT: Final[str] = "EUR_USD"
START_YEAR: Final[int] = 2016
END_YEAR: Final[int] = 2026
LAST_N_DAYS: Final[int] = 90

In [ ]:
from pandas.core.frame import DataFrame
query = (
    f"SELECT * FROM {TABLE} "
    "WHERE instrument = ? AND time >= ? AND time < ? AND complete = 1 "
    "ORDER BY time ASC"
)
with sqlite3.connect(database=DB_PATH) as conn:
    candles: DataFrame = pd.read_sql_query(
        sql=query, con=conn, params=(INSTRUMENT, f"{START_YEAR}-01-01", f"{END_YEAR}-01-01")
    )

# print(candles.head())
print(f"Fetched {len(candles)} {INSTRUMENT} candles from {START_YEAR}-01-01 to {END_YEAR}-01-01 (exclusive).")

In [ ]:
candles["date"] = pd.to_datetime(arg=candles["time"]).dt.date

# Get all unique dates in the candles DF -> sort them chronologically -> take the last N dates of them
last_90_dates = sorted(candles["date"].unique())[-LAST_N_DAYS:]
last_90_days = candles[candles["date"].isin(values=last_90_dates)]

# Get all unique dates in the candles DF -> sort them chronologically -> take the dates from 2016 to 2025 (inclusive)
from_2016_to_2025_dates = sorted(candles["date"].unique())
from_2016_to_2025_days = candles[candles["date"].isin(values=from_2016_to_2025_dates)]

print(f"Last {len(last_90_dates)} trading days: {last_90_dates[0]} → {last_90_dates[-1]}")
print(f"From {START_YEAR} to {END_YEAR} (exclusive) trading days: {from_2016_to_2025_dates[0]} → {from_2016_to_2025_dates[-1]}")

In [ ]:
with pd.option_context("display.max_rows", None, "display.max_columns", None,
                        "display.width", None):
    print(last_90_days.to_string(index=False))

In [ ]:
last_90_days.info()

In [ ]:
last_90_days.shape # output reads: number of rows, number of columns

In [ ]:
last_90_days.head()

In [ ]:
last_90_days.tail()

In [ ]:
week_day_1 = pd.Timestamp("2026-05-31").day_name()  # output reads: Sunday
week_day_2 = pd.Timestamp("2026-07-05").day_name()  # output reads: Sunday
print(week_day_1)
print(week_day_2)

In [ ]:
pd.Timestamp("2025-12-25").day_name() # output reads: Thursday

In [ ]:
ny = pd.to_datetime(last_90_days["time"], utc=True).dt.tz_convert("America/New_York")
session_date = ny.dt.tz_localize(None).dt.normalize() + pd.Timedelta(days=1)

In [ ]:
ny.dt.hour.value_counts()

In [ ]:
session_date.dt.day_name().value_counts()

In [ ]:
session_date